In [7]:
import sys
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error as rmse

df = pd.read_csv("car_fuel_efficiency_2026.csv").fillna(0)
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)
y_train = df_train.pop('fuel_efficiency_mpg').values
y_val = df_val.pop('fuel_efficiency_mpg').values

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val = dv.transform(df_val.to_dict(orient='records'))
names = list(dv.get_feature_names_out())

# Q1
dt = DecisionTreeRegressor(max_depth=1).fit(X_train, y_train)
print('Q1', export_text(dt, feature_names=names))

# Q2
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1).fit(X_train, y_train)
print('Q2', round(rmse(y_val, rf.predict(X_val)), 4))

# Q3 / Q4
for d in [None, 10, 15, 20, 25]:
    scores = []
    for n in [10, 50, 100, 150]:
        m = RandomForestRegressor(n_estimators=n, max_depth=d, random_state=1, n_jobs=-1).fit(X_train, y_train)
        scores.append(round(rmse(y_val, m.predict(X_val)), 3))
    print('Q3/Q4 max_depth', d, scores, 'mean', round(np.mean(scores), 4))

# Q5
rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1).fit(X_train, y_train)
imp = sorted(zip(rf.feature_importances_, names), reverse=True)
print('Q5', [(n, round(i, 4)) for i, n in imp[:6]])

# Q6
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=names)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=names)
for eta in [0.3, 0.1]:
    params = {'eta': eta, 'max_depth': 6, 'min_child_weight': 1, 'objective': 'reg:squarederror',
              'nthread': 8, 'seed': 1, 'verbosity': 1}
    m = xgb.train(params, dtrain, num_boost_round=100, evals=[(dtrain, 'train'), (dval, 'val')], verbose_eval=False)
    print('Q6 eta', eta, round(rmse(y_val, m.predict(dval)), 4))

Q1 |--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]

Q2 1.8332
Q3/Q4 max_depth None [1.833, 1.774, 1.77, 1.769] mean 1.7865
Q3/Q4 max_depth 10 [1.782, 1.748, 1.744, 1.744] mean 1.7545
Q3/Q4 max_depth 15 [1.831, 1.771, 1.768, 1.767] mean 1.7842
Q3/Q4 max_depth 20 [1.837, 1.772, 1.769, 1.768] mean 1.7865
Q3/Q4 max_depth 25 [1.837, 1.772, 1.768, 1.768] mean 1.7862
Q5 [('model_year', np.float64(0.2961)), ('vehicle_weight', np.float64(0.2033)), ('fuel_type=Gasoline', np.float64(0.1607)), ('horsepower', np.float64(0.0783)), ('engine_displacement', np.float64(0.0613)), ('acceleration', np.float64(0.055))]
Q6 eta 0.3 1.8265
Q6 eta 0.1 1.7248
